In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col

# Crear sesión Spark
spark = SparkSession.builder \
    .appName("TransformacionCandidatesKOI") \
    .getOrCreate()


VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1749465031387_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
# Cargar el CSV desde S3
df = spark.read.csv("s3://origendatoskoi/candidates/candidateskoi.csv/part-00000-d37c7f09-7136-4812-a8d3-173b40a6fe1a-c000.csv", header=True, inferSchema=True)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
# Filtrar los campos relevantes
df_filtrado = df.select(
    "kepid",
    "koi_disposition",
    "koi_period",
    "koi_duration",
    "koi_depth",
    "koi_prad",
    "koi_score",
    "koi_teq",
    "koi_srad"
)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
# Renombrar columnas
df_renombrado = df_filtrado.select(
    col("kepid").alias("id_estrella"),
    col("koi_disposition").alias("clasificacion"),
    col("koi_period").alias("periodo_orbital"),
    col("koi_duration").alias("duracion_transito"),
    col("koi_depth").alias("profundidad_transito"),
    col("koi_prad").alias("radio_planeta"),
    col("koi_score").alias("indice_probabilidad_planeta"),
    col("koi_teq").alias("equilibrio_temperatura"),
    col("koi_srad").alias("radio_estrella")
)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
# Clasificación por nivel de probabilidad
df_clasificado = df_renombrado.withColumn(
    "probabilidad_planeta",
    when(col("indice_probabilidad_planeta") >= 0.9, "ALTO")
    .when((col("indice_probabilidad_planeta") >= 0.5) & (col("indice_probabilidad_planeta") < 0.9), "MEDIO")
    .otherwise("BAJO")
)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [7]:
# Ordenar descendente por índice de probabilidad
df_ordenado = df_clasificado.orderBy(col("indice_probabilidad_planeta").desc())


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [9]:
# Guardar el resultado como CSV en S3
df_ordenado.coalesce(1).write.mode("overwrite").option("header", "true").csv("s3://salidatoskoi/transformcandidateskoi.csv")


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…